In [ ]:
import numpy as np
import ufl

from mpi4py import MPI

import dolfinx
import dolfinx.fem.petsc
import dolfinx.mesh
import basix.ufl
from petsc4py import PETSc

length, height = 2., 2.0
Nx, Ny = 2,2
domain = dolfinx.mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([0., 0.]), np.array([length, height])],
    [Nx, Ny],
    cell_type=dolfinx.mesh.CellType.quadrilateral,
)

dim = domain.topology.dim
print(f"Mesh topology dimension d={dim}.")

degree = 2
#shape = (dim,)  # this means we want a vector field of size `dim`
v_elem = basix.ufl.element(
    "Lagrange", 
    domain.topology.cell_name(), 
    degree, 
    shape=(dim,)
)
V = dolfinx.fem.functionspace(domain, v_elem)

u_sol = dolfinx.fem.Function(V, name="Displacement")

E = dolfinx.fem.Constant(domain, 1.)
nu = dolfinx.fem.Constant(domain, 0.)

lmbda = E * nu / (1. + nu) / (1. - 2. * nu)
mu = E / 2. / (1. + nu)

def up_bottom_boundary(x):
    on_bottom = np.isclose(x[1], 0.)
    on_top = np.isclose(x[1], height)
    return on_bottom|on_top
facet_dim = domain.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(domain, facet_dim, up_bottom_boundary)
boundary_tags = dolfinx.mesh.meshtags(
    domain,
    facet_dim,
    boundary_facets,
    np.full(len(boundary_facets), 1, dtype=np.int32)
)

def epsilon(v):
    return ufl.sym(ufl.grad(v))


def sigma(v):
    return lmbda * ufl.tr(epsilon(v)) * ufl.Identity(dim) + 2. * mu * epsilon(v)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

rho = 2e-3
g = 9.81
#f = dolfinx.fem.Constant(domain, np.array([0., -rho * g], dtype=dolfinx.default_scalar_type))
x = ufl.SpatialCoordinate(domain)
#u_exact_x = 0.1*x[0]*(length-x[0])*(ufl.cosh(height)-ufl.cosh(x[1]))
u_exact_x = dolfinx.default_scalar_type(0.0)
epsilon_shift = 1e-5
r =1.# ufl.sqrt((x[0]+epsilon_shift)**2 + (x[1]+epsilon_shift)**2)
#u_exact_y = (ufl.sin(ufl.pi*x[0]/length)**2) * (height-x[1])
u_exact_y = x[0] * (length - x[0]) * (height - x[1])

u_exact = ufl.as_vector((u_exact_x, u_exact_y))
f_mms = -ufl.div(sigma(u_exact))
T_mms = sigma(u_exact)*ufl.FacetNormal(domain)


custom_metadata = {"quadrature_degree": 3}
custom_dx = ufl.Measure("dx", domain=domain, metadata=custom_metadata)
custom_ds = ufl.Measure("ds", domain=domain, subdomain_data=boundary_tags, metadata=custom_metadata)
a = ufl.inner(sigma(u), epsilon(v)) * custom_dx
L = ufl.inner(f_mms, v) * custom_dx + ufl.inner(T_mms, v)*custom_ds(1)

def left(x):
    return np.isclose(x[0], 0.)


def right(x):
    return np.isclose(x[0], length)


left_dofs = dolfinx.fem.locate_dofs_geometrical(V, left)
right_dofs = dolfinx.fem.locate_dofs_geometrical(V, right)
zero_vec = np.zeros(dim, dtype=dolfinx.default_scalar_type)
bcs = [
    dolfinx.fem.dirichletbc(zero_vec, left_dofs, V),
    dolfinx.fem.dirichletbc(zero_vec, right_dofs, V),
]
problem = dolfinx.fem.petsc.LinearProblem(
    a, L, u=u_sol, bcs=bcs,
    petsc_options_prefix="linear_elasticity",
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps"}
)
problem.solve()
# A = dolfinx.fem.petsc.assemble_matrix(a, bcs=bcs)
# A.assemble()

# # Assemble b
# b = dolfinx.fem.petsc.assemble_vector(L)
# dolfinx.fem.petsc.apply_lifting(b, [a], bcs=[bcs])
# b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES,
#               mode=PETSc.ScatterMode.REVERSE)
# dolfinx.fem.petsc.set_bc(b, bcs)
b_lagrange = problem.b

In [ ]:
#np.round(b_lagrange.array, 4)

In [ ]:
# Assuming u_sol is your numerical solution, and u_exact is your reference
# (u_exact must be a dolfinx.fem.Function or a UFL spatial expression)

# Define the error
e = u_sol - u_exact

# 1. L2 Error
error_L2_form = dolfinx.fem.form(ufl.inner(e, e) * custom_dx)
error_L2 = np.sqrt(domain.comm.allreduce(dolfinx.fem.assemble_scalar(error_L2_form), op=MPI.SUM))

# 2. H1 Error
error_H1_form = dolfinx.fem.form((ufl.inner(e, e) + ufl.inner(ufl.grad(e), ufl.grad(e))) * custom_dx)
error_H1 = np.sqrt(domain.comm.allreduce(dolfinx.fem.assemble_scalar(error_H1_form), op=MPI.SUM))

# 3. Energy Error (using your previously defined sigma and epsilon functions)
error_energy_form = dolfinx.fem.form(ufl.inner(sigma(e), epsilon(e)) * custom_dx)
error_energy = np.sqrt(domain.comm.allreduce(dolfinx.fem.assemble_scalar(error_energy_form), op=MPI.SUM))

print(f"L2 Error:     {error_L2:.2e}")
print(f"H1 Error:     {error_H1:.2e}")
print(f"Energy Error: {error_energy:.2e}")
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")


In [ ]:
import pyvista
from dolfinx.plot import vtk_mesh

topology, cell_types, geometry = vtk_mesh(V)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# The solution array is 1D. We reshape it to N x 2 (since dim=2)
u_values = u_sol.x.array.reshape(-1, dim)

# PyVista requires 3D vectors to warp the mesh. We pad the 2D vectors with Z=0.
u_3d = np.zeros((u_values.shape[0], 3), dtype=np.float64)
u_3d[:, :dim] = u_values

# Attach the 3D displacement vectors to the grid
grid.point_data["Displacement"] = u_3d
grid.set_active_vectors("Displacement")

# Warp the grid by the displacement. 
# We use a factor (e.g., 1000) to exaggerate the deformation so it's visible.
warp_factor = 1.0
warped_grid = grid.warp_by_vector("Displacement", factor=warp_factor)

# Plotting
plotter = pyvista.Plotter()
plotter.add_text(f"Deformed Mesh (magnified {warp_factor}x)", font_size=14)

# Show the original undeformed mesh as a wireframe
plotter.add_mesh(grid, style="wireframe", color="black", opacity=0.3, label="Undeformed")

# Show the deformed mesh
plotter.add_mesh(warped_grid, show_edges=False, scalars="Displacement", cmap="coolwarm", label="Deformed")

plotter.view_xy()  # Set camera to view the X-Y plane directly
plotter.show(jupyter_backend="static")

In [ ]:
from THBSplines.src.hierarchical_space import HierarchicalSpace
from THBSplines.src.cartesian_mesh import CartesianMesh
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import cffi
import numba
import numba.core.typing.cffi_utils as cffi_support
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type, geometry
rtype = default_real_type
dtype = default_scalar_type
import ufl
from ffcx.codegeneration.utils import empty_void_pointer
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature

import numpy.typing as npt

def refine(knots: npt.ArrayLike, p: int, n_times: int=1)->npt.NDArray:
    """Given `knots`, returns its dyadic refinement with multiplicity `p+1`
    at the extremities."""
    knots: npt.NDArray[np.float_] = np.asarray(knots)
    mult_left: int = np.searchsorted(knots, knots[0], side='right')
    mult_right: int = len(knots) - np.searchsorted(knots, knots[-1], side='left')
    pad_left: int = max(0, p + 1 - mult_left)
    pad_right: int = max(0, p + 1 - mult_right)
    if pad_left > 0 or pad_right > 0:
        knots = np.concatenate((
            np.full(pad_left, knots[0], dtype=knots.dtype),
            knots,
            np.full(pad_right, knots[-1], dtype=knots.dtype)
        ))
    if n_times == 0:
        return knots
    
    # Find indices where the knot value changes
    jump_idx: npt.NDArray[np.int_] = np.where(knots[1:] > knots[:-1])[0]
    left_vals = knots[jump_idx]
    right_vals =knots[jump_idx + 1]
    num_new_points = (1<<n_times)-1
    fractions = np.linspace(0.,1.,num_new_points+2)[1:-1]
    new_points = left_vals[:, None] + (right_vals - left_vals)[:, None] * fractions[None, :]
    new_points = new_points.ravel()
    insert_positions = np.repeat(jump_idx + 1, num_new_points)
    
    return np.insert(knots, insert_positions, new_points)

p0 = 2
L = 2.
h=1.
n_refinements = 0
knotsx = np.array([0., L/2., L], dtype=np.float64)
knotsx = refine(knotsx, p=p0, n_times=n_refinements)
#log_initial_mesh_size = np.log2(np.max(np.diff(knotsx)))
knotsy = refine(np.array([0., h/2., h], dtype=np.float64), p=p0, n_times=0)
err_cells = {}
hs = HierarchicalSpace(knots=[knotsx, knotsy], degrees=[p0])
        

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=True)
hs.hmesh.plot_cells()

In [ ]:
total_active_cells = sum(len(hs.hmesh.aelem_level[l]) for l in range(hs.nlevels))

all_cells = np.empty((2**hs.dim*total_active_cells, hs.dim), dtype=np.float64) # will have coarser cells on top and finer on bottom
thb_operators: dict[tuple[int, int], npt.NDArray[np.float64]] = {}
N_max = 0 # maximum amount of dofs in a cell
current_idx = 0
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    if len(active_cells_l)==0:
        continue
   
    thb_operators_list = hs.local_multi_level_extraction_operator2(active_cells_l, l, l)
    thb_operators.update({(l, cell): op for cell, op in zip(active_cells_l, thb_operators_list)})
    
    if thb_operators_list:
        level_max = max(op.shape[0] for op in thb_operators_list)
        N_max = max(N_max, level_max)

    mesh = CartesianMesh(hs.hmesh.one_d_indices[l], len(hs.hmesh.one_d_indices[l]))
    my_cells_l = mesh.cells[active_cells_l]

    n_cells = len(my_cells_l)
    x_coords = my_cells_l[:, 0, :]
    y_coords = my_cells_l[:, 1, :] 

    start, end = current_idx, current_idx+(2**hs.dim*n_cells)

    view = all_cells[start:end]
    view[0::4] = np.column_stack((x_coords[:, 0], y_coords[:, 0]))  # (xmin, ymin)
    view[1::4] = np.column_stack((x_coords[:, 1], y_coords[:, 0]))  # (xmax, ymin)
    view[2::4] = np.column_stack((x_coords[:, 0], y_coords[:, 1]))  # (xmin, ymax)
    view[3::4] = np.column_stack((x_coords[:, 1], y_coords[:, 1]))  # (xmax, ymax)
    current_idx=end
pass
del mesh # free this big object
all_cells = np.array(all_cells).reshape(-1, hs.dim)

coordinates = np.arange(len(all_cells), dtype=np.int32).reshape(-1, 2**hs.dim)
coordinate_element = basix.ufl.element("Q", "quadrilateral", 1, shape=(hs.dim,))
disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells=coordinates, e=coordinate_element, x=all_cells)
del all_cells
del coordinates
#del thb_operators_list

In [ ]:
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
def up_bottom_boundary(x):
    on_bottom = np.isclose(x[1], knotsy[0])
    on_top = np.isclose(x[1], knotsy[-1])
    return on_bottom|on_top

def left_right_boundaries(x):
    on_left = np.isclose(x[0], knotsx[0])
    on_right = np.isclose(x[0], knotsx[-1])
    return on_left|on_right

dim = disconnected_mesh.topology.dim
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    shape=(dim,),
    lagrange_variant=basix.LagrangeVariant.legendre
)

V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
# print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
facet_dim = disconnected_mesh.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, up_bottom_boundary)
boundary_tags = dolfinx.mesh.meshtags(
    disconnected_mesh,
    facet_dim,
    boundary_facets,
    np.full(len(boundary_facets), 1, dtype=np.int32)
)
custom_metadata = {"quadrature_degree": 3}
ds_custom = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=boundary_tags, metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)
#u_sol = dolfinx.fem.Function(V, name="Displacement")

#E = dolfinx.fem.Constant(disconnected_mesh, 210e3)
#nu = dolfinx.fem.Constant(disconnected_mesh, 0.3)

#lmbda = E * nu / (1. + nu) / (1. - 2. * nu)
E=1.
nu = 0.3
lmbda = E*nu/(1.+nu)/(1.-2.*nu)
#mu = E / 2. / (1. + nu)
mu = E/2./(1.+nu)
lmbda_c = dolfinx.fem.Constant(disconnected_mesh, lmbda)
mu_c = dolfinx.fem.Constant(disconnected_mesh, mu)

def epsilon(v):
    return ufl.sym(ufl.grad(v))

def sigma(v):
    return lmbda_c * ufl.tr(epsilon(v)) * ufl.Identity(dim) + 2. * mu_c * epsilon(v)


u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
rho = 2e-3
g = 9.81
my_x = ufl.SpatialCoordinate(disconnected_mesh)

#u_exact_x = dolfinx.default_scalar_type(0.0)
u_exact_x = 0.3*my_x[0] * (L - my_x[0]) * (-2.- 1.4*my_x[1])
#epsilon_shift = 1e-5
#r =1.# ufl.sqrt((x[0]+epsilon_shift)**2 + (x[1]+epsilon_shift)**2)
#u_exact_y = (ufl.sin(ufl.pi*x[0]/L)**2) * (l-x[1])
u_exact_y = my_x[0] * (L - my_x[0]) * (1.+ my_x[1])
u_exact = ufl.as_vector((u_exact_x, u_exact_y))
f_mms = -ufl.div(sigma(u_exact))
T_mms = sigma(u_exact)* ufl.FacetNormal(disconnected_mesh)

a = ufl.inner(sigma(u), epsilon(v)) * dx_custom
L_cell = ufl.inner(f_mms, v) * dx_custom 
L_facet = ufl.inner(T_mms, v)*ds_custom(1) 


msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcx_L_cell, _, _ = ffcx_jit(msh.comm, L_cell, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernel_L_cell = getattr(ufcx_L_cell.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcx_L_facet, _, _ = ffcx_jit(msh.comm, L_facet, form_compiler_options={"scalar_type": dtype})
kernel_L_facet = getattr(ufcx_L_facet.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")

ffi = cffi.FFI()

In [ ]:
#V.dofmap.cell_dofs(10)

In [ ]:
dofmap, dummy_dof_index = hs.build_global_dof_map()
dummy_dof_index += 1 
num_control_points_with_dummy = dummy_dof_index+1

num_cells = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_global

# which BSpline functions are active on each cell, padded with the dummy index.
padded_cells_to_dofs = np.full((num_cells, N_max), dummy_dof_index, dtype=np.int32)

# Find the maximum local index for each level to size arrays
max_local_indices = [0] * hs.nlevels
for l, local_idx in dofmap.keys():
    if local_idx > max_local_indices[l]:
        max_local_indices[l] = local_idx
    pass
pass

# convert the dictionary into a list of numpy arrays for faster lookup
dofmap_arrays = [np.full(size + 1, dummy_dof_index, dtype=np.int32) for size in max_local_indices]
for (l, local_idx), global_idx in dofmap.items():
    dofmap_arrays[l][local_idx] = global_idx


cell_index = 0
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    for cell in active_cells_l:
        
        # Get the local functions on the cell
        active_funcs_dict: dict[int, npt.NDArray[np.int_]] = hs.get_all_active_functions_on_cell(l, cell)
            
        #  Vectorized conversion from local to global indices
        mapped_arrays = [
            dofmap_arrays[ll][local_funcs] for ll, local_funcs in active_funcs_dict.items() if len(local_funcs) > 0
        ]
        
        if mapped_arrays: # Check if there are actually active functions
            global_dofs = np.concatenate(mapped_arrays)
            n_dofs = len(global_dofs)
            padded_cells_to_dofs[cell_index, :n_dofs] = global_dofs
            
        cell_index += 1
new_pctd = padded_cells_to_dofs
# for i in range(len(new_pctd)):
#     row = new_pctd[i, :]
#     no_dummy = row[row<dummy_dof_index]
#     my_median = int(np.median(no_dummy))
#     new_pctd[i, :][new_pctd[i, :]==dummy_dof_index] = my_median


In [ ]:
pctd_size = padded_cells_to_dofs.size
#new_pctd = np.arange(pctd_size).reshape(padded_cells_to_dofs.shape)
print(new_pctd)

In [ ]:
M = hs._bezier_to_legendre(degree = p0)
S_indices = np.arange(p0+1, dtype=np.float64)
# scaling for unnormalised Legendre basis polynomials
S_inv = (1./np.sqrt(2.*S_indices+1.))*np.identity(p0+1, dtype=np.float64) # Scaling factor, since fenicsx uses orthonormal legendre polynomials
T = np.asfortranarray(np.kron(M, M).T @ np.kron(S_inv, S_inv), dtype=dtype)
local_dofs_size = T.shape[1]

operator_shape = (N_max, local_dofs_size)
# Create a custom space that holds the content of each matrix for the relevant cell.
# degree 0 because the value is constant over each cell
C_space = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0, operator_shape))
C_func = dolfinx.fem.Function(C_space, dtype=dtype)

num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local
indices = np.arange(num_cells_local, dtype=np.int32)
# To make sure that each matrix is assigned to the correct cell
midpoints: npt.NDArray[np.float_] = dolfinx.mesh.compute_midpoints(disconnected_mesh, disconnected_mesh.topology.dim, indices)

c_values = C_func.x.array.reshape((-1, N_max, local_dofs_size))
for local_idx, midpoint in enumerate(midpoints):
    #print(f"midpoint = {midpoint[:2]}")
    level, idx = hs.hmesh.find_active_cell(midpoint[:hs.dim])
    #level, idx = find_level_idx_from_midpoint(hs, midpoint=midpoint[:hs.dim], all_midpoints=physical_cells_midpoints)
    #print(f"level={level}, idx={idx}\n")

    mat = thb_operators[level, idx] @ hs.level_spaces[level].get_bezier_operator(idx)
    
    real_k, n_cols = mat.shape

    if real_k<N_max:
        padding_size = N_max - real_k
        to_pad = np.zeros((padding_size, mat.shape[1]), dtype=np.float64)
        #mat_padded = np.vstack((mat, np.zeros((padding_size, mat.shape[1])) ))
        Ci = mat
    else:
        Ci = mat
        to_pad = np.zeros((0, mat.shape[1]))
    
    # is a view of C_func.x.array, therefore we modify the content of C_func.x.array
    # No new array is created, the matrix->cell mapping is done here.
    c_values[local_idx, :, :] = np.vstack((Ci@T, to_pad))
C_func.x.scatter_forward()

In [ ]:
num_control_points = np.max(new_pctd)+1
my_index_map = dolfinx.common.IndexMap(comm=disconnected_mesh.comm, 
                                       local_size=2*num_control_points)

dummy_element = basix.ufl.element(
    family="DG", 
    cell="quadrilateral", 
    degree=0, 
    shape=(2*N_max,)
)
dummy_space = dolfinx.fem.functionspace(mesh=disconnected_mesh, element=dummy_element)
element_layout = dummy_space.dofmap.dof_layout
cpp_element = dummy_space.element._cpp_object

vector_dofs = np.zeros((num_cells, 2*N_max), dtype=np.int32)
vector_dofs[:, 0::2] = 2 * new_pctd       # X DOFs
vector_dofs[:, 1::2] = 2 * new_pctd + 1   # Y DOFs

dof_indices = vector_dofs.ravel().astype(np.int32)
offsets = (np.arange(num_cells + 1, dtype=np.int32) * 2*N_max).astype(np.int32)

# interleave indices as {0:np.array([x_{00}, y_{00}, x_{01}, y_{01}, ..., x_{0Nmax}, y_{0Nmax}]),
#                        1:np.array([x_{10}, y_{10}, x_{11}, y_{11}, ..., x_{1Nmax}, y_{1Nmax}]),
#                        ...,
#                        Ncells-1:np.array([x_{..0}, y_{..0}, x_{..1}, y_{..1}, ..., x_{..Nmax}, y_{..Nmax}])}
adj = dolfinx.cpp.graph.AdjacencyList_int32(data=dof_indices, 
                                            offsets=offsets)

doflinx_dofmap = dolfinx.cpp.fem.DofMap(
    element_dof_layout=element_layout, 
    index_map=my_index_map,  
    index_map_bs=1, 
    dofmap=adj, 
    bs=1
)

V_spline_cpp = dolfinx.cpp.fem.FunctionSpace_float64(
    mesh=disconnected_mesh._cpp_object, 
    element=cpp_element, 
    dofmap=doflinx_dofmap
)

V_spline = dolfinx.fem.FunctionSpace(
    mesh=disconnected_mesh, 
    element=dummy_element, 
    cppV=V_spline_cpp
)

In [ ]:
adj

In [ ]:
aaa = np.array([[1,2,3], [4,5,6], [7,8,9]])
bbb = np.zeros((2*aaa.shape[0], 2*aaa.shape[1]), dtype=int)
for i in range(aaa.shape[0]):
    for j in range(aaa.shape[1]):
        bbb[2*i,2*j] = aaa[i,j]
        bbb[2*i+1, 2*j+1] = aaa[i,j]
print(bbb)

In [ ]:
PADDED_DOFS = N_max
LOCAL_DOFS = local_dofs_size

LOCAL_DOFS_VEC = 2 * local_dofs_size
PADDED_DOFS_VEC = 2 * N_max

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)  # type: ignore
def tabulate_A(A_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):

    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # returns a view over the original array A_
    # This has to be larger since we are working with padded arrays. Irrelevant dofs are mapped to a dummy location
    A = numba.carray(A_, (PADDED_DOFS_VEC, PADDED_DOFS_VEC), dtype=dtype)

    # Get the operator (TRUNC @ C_{B->BS} @ (C_{L->B}.T) @ S^{-1}) for this cell
    # TRUNC has shape (PADDED_DOFS, LOCAL_DOFS) and all other matrices have shape (LOCAL_DOFS, LOCAL_DOFS)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)
    
    # Tabulate all sub blocks locally
    # This matrix is formed via the Legendre elements on a quadrilateral of degree p0,
    # therefore this has the shape (LOCAL_DOFS, LOCAL_DOFS)
    A0 = np.zeros((LOCAL_DOFS_VEC, LOCAL_DOFS_VEC), dtype=dtype)
    kernela0(
        ffi.from_buffer(A0),
        w_,
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )

    G_vec = np.zeros((PADDED_DOFS_VEC, LOCAL_DOFS_VEC), dtype=dtype)
    for i in range(PADDED_DOFS):
        for j in range(LOCAL_DOFS):
            G_vec[2*i, 2*j]     = G[i, j] # X-component mapping
            G_vec[2*i+1, 2*j+1] = G[i, j] # Y-component mapping
        pass
    pass
            
    
    A[:, :] = G_vec@A0@(G_vec.T) 

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_L_cell(b_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    
    # Prepare target condensed local element tensor
    b = numba.carray(b_, (PADDED_DOFS_VEC,), dtype=dtype)

    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)

    b0 = np.zeros((LOCAL_DOFS_VEC,), dtype=dtype)
    kernel_L_cell(ffi.from_buffer(b0),
        w_,
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
    G_vec = np.zeros((PADDED_DOFS_VEC, LOCAL_DOFS_VEC), dtype=dtype)
    for i in range(PADDED_DOFS):
        for j in range(LOCAL_DOFS):
            G_vec[2*i, 2*j]     = G[i, j]
            G_vec[2*i+1, 2*j+1] = G[i, j]
        pass
    pass
            
   
    b[:] = G_vec@b0

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_L_facet(b_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    b = numba.carray(b_, (PADDED_DOFS_VEC,), dtype=dtype)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)

    b0 = np.zeros((LOCAL_DOFS_VEC,), dtype=dtype)
    kernel_L_facet(ffi.from_buffer(b0), 
                   w_, 
                   c_, 
                   coords_, 
                   entity_local_index, 
                   permutation, 
                   empty_void_pointer())
   
    
    G_vec = np.zeros((PADDED_DOFS_VEC, LOCAL_DOFS_VEC), dtype=dtype)
    for i in range(PADDED_DOFS):
        for j in range(LOCAL_DOFS):
            G_vec[2*i, 2*j]     = G[i, j]
            G_vec[2*i+1, 2*j+1] = G[i, j]
            
    b[:] = G_vec @ b0

In [ ]:
facet_dim = msh.topology.dim-1

boundary_facets = dolfinx.mesh.locate_entities_boundary(msh, facet_dim, up_bottom_boundary)
msh.topology.create_connectivity(facet_dim, msh.topology.dim)
msh.topology.create_connectivity(msh.topology.dim, facet_dim)

# dictionary of facet->cell
f_to_c = msh.topology.connectivity(facet_dim, msh.topology.dim)
# dictionary of cell->array[facets]
c_to_f = msh.topology.connectivity(msh.topology.dim, facet_dim)

boundary_entities = []
# Loop over all edges that belong to our exterior
for f in boundary_facets:
    # returns the cells linked to this edge
    cells = f_to_c.links(f)
    # Exterior facets only have 1 attached cell, 
    # hence get the first one 
    c = cells[0] 
    
    # Find the local index (e.g., 0, 1, 2, or 3 for quads) of facet f within cell c
    local_f = np.where(c_to_f.links(c) == f)[0][0]
    #print(f"f={f}, cell={c}, local_f = {local_f}")
    
    boundary_entities.extend([c, local_f])

# FEniCSx custom form arrays must be typed as int32
boundary_entities = np.array(boundary_entities, dtype=np.int32)

In [ ]:
cpp_constants = [lmbda_c._cpp_object, mu_c._cpp_object]
#cpp_constants = []
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=cpp_constants,
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_L_cell.address, cells, np.array([0], dtype=np.int8))],
                 dolfinx.fem.IntegralType.exterior_facet: [(0, tabulate_L_facet.address, boundary_entities, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of stuff to integrate
        coefficients=[C_func._cpp_object], # holds C@T
        constants=cpp_constants, need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
# boundary_entities.reshape(-1, 2)

In [ ]:
def get_spline_indices_left_right(hs, dofmap):
    dirichlet_indices = {}
    for level in range(hs.nlevels):
        hs.level_spaces[level].construct_basis()
        basis = hs.level_spaces[level].basis
        right_dirichlet_indices = np.isclose(basis[:, 1, -hs.degrees[0]-1:], np.full((hs.degrees[0]+1), fill_value=L, dtype=np.float64))
        right_dirichlet_indices = np.all(right_dirichlet_indices, axis=-1)
        right_dirichlet_indices = np.nonzero(right_dirichlet_indices)[0]

        left_dirichlet_indices = np.isclose(basis[:, 1, :-1], np.zeros((hs.degrees[0]+1), dtype=np.float64))
        left_dirichlet_indices = np.all(left_dirichlet_indices, axis=-1)
        left_dirichlet_indices = np.nonzero(left_dirichlet_indices)[0]

        # (A\cap B)\cup(A\cap C) = A\cap(B\cup C)
        dirichlet_indices[level] = np.intersect1d(hs.truly_active[level],
                                                  np.union1d(right_dirichlet_indices,left_dirichlet_indices),
                                                  assume_unique=True)
    pass
    forbidden_indices = np.array([dofmap[level,idx] for level in dirichlet_indices 
              for idx in dirichlet_indices[level]],
                        dtype=np.int32)
    return forbidden_indices
forbidden_indices = get_spline_indices_left_right(hs, dofmap)
if forbidden_indices is not None and len(forbidden_indices) > 0:
    forbidden_indices_vec = np.empty(2 * len(forbidden_indices), dtype=np.int32)
    forbidden_indices_vec[0::2] = 2 * forbidden_indices      # X DOFs
    forbidden_indices_vec[1::2] = 2 * forbidden_indices + 1  # Y DOFs
    forbidden_indices = forbidden_indices_vec

In [ ]:
#hs.level_spaces[0].basis

In [ ]:
# print(forbidden_indices)

In [ ]:
from dolfinx.fem.petsc import assemble_matrix, assemble_vector
from petsc4py import PETSc
# a_form = dolfinx.fem.form(a0)
A = assemble_matrix(a_cond, bcs=[])
A.assemble()
one_active=False
two_active= False
for level in range(hs.nlevels):
    if level in hs.truly_active and hs.truly_active[level].size>0:
        if one_active:
            two_active=True
        one_active=True

A_mat = A
if two_active:
    dummy_x = 2 * dummy_dof_index
    dummy_y = 2 * dummy_dof_index + 1
    A_mat.setValue(dummy_x, dummy_x, 1., addv=PETSc.InsertMode.INSERT_VALUES)
    A_mat.setValue(dummy_y, dummy_y, 1., addv=PETSc.InsertMode.INSERT_VALUES)
    A_mat.assemble()
    A_mat.assemblyBegin()
    A_mat.assemblyEnd()

b = assemble_vector(l_cond)
if two_active:
    b[dummy_x] = 0.0
    b[dummy_y] = 0.0
    b.assemblyBegin()
    b.assemblyEnd()

if forbidden_indices is not None:
    A_mat.zeroRowsColumns(forbidden_indices, diag=1.0, x=None, b=b)
    b.array_w[forbidden_indices]=0.
b.ghostUpdate(addv=PETSc.InsertMode.INSERT, mode=PETSc.ScatterMode.FORWARD)
#print(f"Two active = {two_active}")


In [ ]:
print(np.round(b.array, 4))

In [ ]:
# from petsc4py import PETSc

# A_trans = A_mat.transpose()
# # A_diff = A - A^T
# A_diff = A_mat.duplicate()
# A_mat.copy(A_diff)
# A_diff.axpy(-1.0, A_trans)

# # Get the maximum non-symmetric value
# norm_diff = A_diff.norm(PETSc.NormType.INFINITY)
# print(f"Matrix Asymmetry (||A - A^T||_inf): {norm_diff:.2e}")

In [ ]:
# # Extract the CSR (Compressed Sparse Row) arrays from PETSc
# indptr, indices, data = A_mat.getValuesCSR()

# # Get the global size of the matrix
# shape = A_mat.getSize()

# # Create a SciPy CSR matrix
# A_scipy = sp.csr_array((data, indices, indptr), shape=shape)

# print(f"Matrix shape: {A_scipy.shape}")
# print(f"Number of non-zeros: {A_scipy.nnz}")

# import matplotlib.pyplot as plt

# plt.figure(figsize=(7, 7))
# # plt.spy plots the non-zero entries of a matrix
# plt.spy(A_scipy, markersize=2, color='lightseagreen')
# plt.title("Sparsity Pattern of THB-Spline mass Matrix")
# plt.show()

In [ ]:
#print(f"{np.linalg.cond(A_scipy.toarray()):.3e}")

In [ ]:
ksp = PETSc.KSP().create(A_mat.comm)
ksp.setOperators(A_mat)
#ksp.setType(PETSc.KSP.Type.CG)
#ksp.getPC().setType(PETSc.PC.Type.JACOBI)
ksp.setType(PETSc.KSP.Type.PREONLY)
ksp.getPC().setType(PETSc.PC.Type.LU)
ksp.getPC().setFactorSolverType("mumps")
u_sol = dolfinx.fem.Function(V_spline)
ksp.solve(b, u_sol.x.petsc_vec)
u_sol.x.scatter_forward()
x_vec=u_sol.x.array
print(f"Solve complete. Reason: {ksp.getConvergedReason()}, Iterations: {ksp.getIterationNumber()}")

In [ ]:
# Map the global B-spline coefficients back to local Legendre coefficients
u_dg = dolfinx.fem.Function(V)
block_size = V.dofmap.bs

# x_vec is the solution vector
for local_idx in range(msh.topology.index_map(msh.topology.dim).size_local):
    spline_dofs = vector_dofs[local_idx]
    u_spline_local = x_vec[spline_dofs]
    
    G = c_values[local_idx, :, :]
    G_vec = np.zeros((2*G.shape[0], 2*G.shape[1]), dtype=dtype)
    for i in range(G.shape[0]):
        for j in range(G.shape[1]):
            G_vec[2*i, 2*j]     = G[i, j] # X-component mapping
            G_vec[2*i+1, 2*j+1] = G[i, j] # Y-component mapping
        pass
    pass
    u_dg_local = G_vec.T @ u_spline_local
    
    dg_dofs = V.dofmap.cell_dofs(local_idx)
    unrolled_dg_dofs = np.empty(len(dg_dofs) * block_size, dtype=np.int32)
    for i in range(block_size):
        unrolled_dg_dofs[i::block_size] = dg_dofs * block_size + i
        
    u_dg.x.array[unrolled_dg_dofs] = u_dg_local


u_dg.x.scatter_forward()


e = u_dg-u_exact

# Compute L2 Error: sqrt( \int (u_bar - u_dg)^2 dx )
error_L2_form = dolfinx.fem.form(ufl.inner(e,e) * dx_custom)
error_L2_sq = dolfinx.fem.assemble_scalar(error_L2_form)
l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_L2_sq, op=MPI.SUM))

# Compute H1 Semi-norm (Gradient) Error: sqrt( \int |grad(u_bar) - grad(u_dg)|^2 dx )
# This is the "energy" error and is crucial for elliptic PDEs!
error_H1_form = dolfinx.fem.form((ufl.inner(e,e)+ufl.inner(ufl.grad(e), ufl.grad(e))) * dx_custom)
error_H1_sq = dolfinx.fem.assemble_scalar(error_H1_form)
h1_error = np.sqrt(disconnected_mesh.comm.allreduce(error_H1_sq, op=MPI.SUM))

# 3. Energy Error (using your previously defined sigma and epsilon functions)
error_energy_form = dolfinx.fem.form(ufl.inner(sigma(e), epsilon(e)) * dx_custom)
error_energy = np.sqrt(disconnected_mesh.comm.allreduce(dolfinx.fem.assemble_scalar(error_energy_form), op=MPI.SUM))


# Print results
print(f"Absolute L2 Error: {l2_error:.2e}")
#print(f"Relative L2 Error: {l2_error / exact_L2_norm:.2e}\n")

print(f"Absolute H1 Error: {h1_error:.2e}")
#print(f"Relative H1 Error: {h1_error / exact_H1_norm:.2e}")
print(f"Energy Error: {error_energy:.2e}")
print(f"dofs = {A_mat.getSize()[0]-2}")
#ksp.destroy()
#A.destroy()
#b.destroy()

In [ ]:
# V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
# v = ufl.TestFunction(V_error)

# #hQ = ufl.CellDiameter(disconnected_mesh)
# #volume_form = dolfinx.fem.form(1.0*v*dx_custom)
# #cell_volumes = dolfinx.fem.assemble_vector(volume_form).array

# # Define the local L2 error form: integral of (f - u_dg)^2 per cell
# # Note: We multiply by the test function 'v' to pick out each cell's contribution
# e = u_dg-u_exact
# local_error_form = dolfinx.fem.form(ufl.inner(sigma(e), epsilon(e)) * v * dx_custom)

# # Assemble the vector (this gives us the squared error per cell)
# local_error_vector = dolfinx.fem.assemble_vector(local_error_form)
# local_error_vector.scatter_forward()

# cell_errors = np.sqrt(local_error_vector.array)
# squared_errors = cell_errors**2
# total_squared_error = np.sum(squared_errors)
# descending_indices = np.flip(np.argsort(squared_errors))
# sorted_squared_errors = squared_errors[descending_indices]
# cumulative_errors = np.cumsum(sorted_squared_errors)
# theta = 0.2
# threshold_value = theta*total_squared_error
# num_cells_to_mark = max(2, np.searchsorted(cumulative_errors, threshold_value)+1)
# top_error_indices = descending_indices[:num_cells_to_mark]
# print(f"Total cells marked via Dörfler (theta={theta}): {num_cells_to_mark} out of {len(cell_errors)}")
# print(f"Indices to refine: {top_error_indices[:10]}")

# my_arr = []
# for value in hs.hmesh.aelem_level.values():
#     my_arr.extend(value)

# err_cells = {}
# sorted_top_error_indices = np.sort(top_error_indices)
# level=0
# current_length = len(hs.hmesh.aelem_level[0])
# for i in sorted_top_error_indices:
#     while i > current_length-1:
#         level+=1
#         current_length+=len(hs.hmesh.aelem_level[level])
#     pass
#     if level not in err_cells:
#         err_cells[level]=[]
    
#     err_cells[level].append(my_arr[i])
# pass

In [ ]:
# import dolfinx
# from mpi4py import MPI
# import basix.ufl
# import numpy as np

# # 1. Recreate your exact element
# mesh = dolfinx.mesh.create_unit_square(MPI.COMM_WORLD, 1, 1, cell_type=dolfinx.mesh.CellType.quadrilateral)
# legendre_elt = basix.ufl.element(
#     "DG", "quadrilateral", degree=2, shape=(2,), 
#     lagrange_variant=basix.LagrangeVariant.legendre
# )
# V = dolfinx.fem.functionspace(mesh, legendre_elt)

# # 2. Interpolate a vector where X=10.0 and Y=20.0
# u_test = dolfinx.fem.Function(V)
# u_test.interpolate(lambda x: (np.full_like(x[0], 10.0), np.full_like(x[0], 20.0)))

# # 3. Print the array
# print(f"{np.round(u_test.x.array, 3)}")

In [ ]:
# import numpy as np
# from scipy.interpolate import BSpline
# import scipy.integrate as integrate

# # --- 1. Problem Setup ---
# L, l = 2.0, 2.0
# p = 2
# knots = [0.0, 0.0, 0.0, 1.0, 2.0, 2.0, 2.0]
# n_cp = len(knots) - p - 1  # 4 control points per direction

# # Helper to evaluate 1D B-spline basis function i at point x
# def B(i, x_val):
#     c = np.zeros(n_cp)
#     c[i] = 1.0
#     val = BSpline(knots, c, p, extrapolate=False)(x_val)
#     return np.nan_to_num(val) # handle values strictly outside support

# # --- 2. Forcing and Tractions ---
# def f_x(x, y): return 1.0 - x
# def f_y(x, y): return 4.0 - 2.0*x - y

# def T_top_x(x): return 0.0
# def T_top_y(x): return x**2 - 2.0*x

# def T_bot_x(x): return 2.0*x - 2.0
# def T_bot_y(x): return 2.0*x - x**2

# # --- 3. Exact Integration ---
# b_exact = np.zeros((n_cp * n_cp * 2,))

# for i in range(n_cp):      # X-basis index
#     for j in range(n_cp):  # Y-basis index
        
#         # Helper integrands for 2D body forces
#         def integrand_fx(x, y): return f_x(x, y) * B(i, x) * B(j, y)
#         def integrand_fy(x, y): return f_y(x, y) * B(i, x) * B(j, y)
        
#         vol_int_x, _ = integrate.nquad(integrand_fx, [[0, L], [0, l]])
#         vol_int_y, _ = integrate.nquad(integrand_fy, [[0, L], [0, l]])
        
#         # Boundary integrations
#         top_int_x, _ = integrate.quad(lambda x: T_top_x(x) * B(i, x) * B(j, 2.0), 0, L)
#         top_int_y, _ = integrate.quad(lambda x: T_top_y(x) * B(i, x) * B(j, 2.0), 0, L)
        
#         bot_int_x, _ = integrate.quad(lambda x: T_bot_x(x) * B(i, x) * B(j, 0.0), 0, L)
#         bot_int_y, _ = integrate.quad(lambda x: T_bot_y(x) * B(i, x) * B(j, 0.0), 0, L)
        
#         # Assemble local entries
#         total_bx = vol_int_x + top_int_x + bot_int_x
#         total_by = vol_int_y + top_int_y + bot_int_y
        
#         # Dof mapping: assuming ordering [X0, Y0, X1, Y1, ...] going row by row
#         dof_idx = (i * n_cp + j) * 2
#         b_exact[dof_idx]     = total_bx
#         b_exact[dof_idx + 1] = total_by

# # Print the non-zero structure to compare
# print("Analytical b-vector (Interleaved: X0, Y0, X1, Y1...):")
# for idx in range(0, len(b_exact), 2):
#     print(f"CP({idx//8}, {(idx//2)%4}) | b_x: {b_exact[idx]:>8.4f} | b_y: {b_exact[idx+1]:>8.4f}")